# Advanced RAG 핵심 실습
* Day 17에서 배운 내용 중 **꼭 손으로 한 번 볼 것**만 빠르게 확인합니다.
* 16.4에서 만든 Chroma(학부·대학원 학칙)를 그대로 이어 씁니다.



---
* `day15` Conda 환경을 사용합니다.
* 오늘 핵심만 다룹니다.

| 실습 | 슬라이드 개념 |
|------|----------------|
| Query Rewriting | 모호한 질문을 검색어로 다시 쓰기 |
| Multi-Query | 여러 표현으로 검색 후 통합 |
| CRAG식 3단계 판단 | Correct / Ambiguous / Incorrect |
| 미니 골든셋 | Precision을 숫자로 보기 |




In [10]:
!pip install langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langgraph pypdf chromadb -q

In [11]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
DAY16 = WORKDIR.parent / '16일차'
RAG_DIR = DAY16 / 'rag_data'
CHROMA_DIR = DAY16 / 'chroma_regulations'

print('WORKDIR :', WORKDIR)
print('CHROMA  :', CHROMA_DIR, '| exists:', CHROMA_DIR.exists())
print('PDF 수  :', len(list(RAG_DIR.glob('*.pdf'))))



WORKDIR : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/17일차
CHROMA  : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/16일차/chroma_regulations | exists: True
PDF 수  : 2


---
## 0. 기존 Chroma 불러오기


In [12]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings(model='text-embedding-3-small', api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=OPENAI_API_KEY)

if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f'{CHROMA_DIR} 없음. 먼저 16.4 노트북에서 Chroma 인덱싱을 완료하세요.'
    )

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding,
)
print('로드 완료. 예시 검색:')
for d in vectorstore.similarity_search('석사 수업연한', k=2):
    print('-', d.metadata.get('level'), '|', d.page_content[:80].replace('\n', ' '))



로드 완료. 예시 검색:
- 학부 | 서 지정한 한국어 능력을 취득한 자 <개정 2021. 10. 13.> 10. 졸업연기를 원하는 경우 정해진 기일 내에 졸업연기신청을 하여야 하며
- 학부 | ③ 학사학위수료자의 재학연한은 재학과 학사학위수료 기간을 포함하며, 재학연한이 만료되 는 경우 졸업생으로 변경한다. <조신설 2019.01.28


---
## 1. Baseline — 모호한 질문 그대로 검색
* 구어체·모호한 질문은 검색 엔진이 잘 못 알아듣습니다.
* **재작성 전** 결과를 먼저 봅니다.



In [13]:
RAW_Q = '졸업하려면 대충 몇 년 다니면 되는 거야? 석사 쪽'

baseline = vectorstore.similarity_search(RAW_Q, k=3, filter={'level': '대학원'})
print('질문:', RAW_Q)
for i, d in enumerate(baseline, 1):
    print(f'[{i}]', d.page_content[:160].replace('\n', ' '))
    print('---')



질문: 졸업하려면 대충 몇 년 다니면 되는 거야? 석사 쪽
[1] 2월 이전에 입학한 학생은 본 학칙 제15조에 불구하고 석사학위과정의 경우 24학점 이상 취득하여야 한다.2. 1974년 2월 이전에 소요학점의 이수를 필하고 2학기 이상 연구등록을 한 학생은 본 학칙 제28조에 불구하고 종합시험의 합격과 동시에 학위논문을 제출할 수 있다.
---
[2] 제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은
---
[3] 연협력촉진에 관한 법률」 제8조 및 같은 법 시행령 제8조, 제9조에 따라 계약에 의하여 운영되는 대학원 교육과정에 입학하는 학생 제3장 입학제4조(입학 및 전형시기) ① 대학원의 입학(편입학 및 재입학을 포함한다) 시기는 매 학기 시작일 이내로 한다.② 입학전형은 일반전형과 특별전형으
---


---
## 2. Query Rewriting
* 원래 질문을 LLM이 **검색에 적합한 문장**으로 다시 씁니다.
* 재작성된 질문만 Retriever에 넣습니다.



In [14]:
def rewrite_query(question: str) -> str:
    prompt = (
        '다음 사용자 질문을 학칙 문서 검색에 적합한 짧고 명확한 한국어 검색어로 다시 쓰세요.\n'
        '핵심 키워드(학위과정, 학부/대학원, 수업연한 등)는 유지하세요.\n'
        '검색어만 한 줄로 출력하세요.\n\n'
        f'질문: {question}'
    )
    return llm.invoke(prompt).content.strip()


rewritten = rewrite_query(RAW_Q)
print('원래   :', RAW_Q)
print('재작성 :', rewritten)



원래   : 졸업하려면 대충 몇 년 다니면 되는 거야? 석사 쪽
재작성 : 석사 학위과정 수업연한


In [15]:
rewritten_docs = vectorstore.similarity_search(rewritten, k=3, filter={'level': '대학원'})
print('재작성 검색 결과:')
for i, d in enumerate(rewritten_docs, 1):
    print(f'[{i}]', d.page_content[:160].replace('\n', ' '))
    print('---')



재작성 검색 결과:
[1] ③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단,
---
[2] 한 교원에 대하여 대학원운영위원회의 심의·의결을 거쳐 교원인사위원회에 회부하여야 한다.제21조(이수인정 평점) 학생의 수강과목별 출석율이 한 학기 간 실제 수업 기간 수의 3분의 2 이상이고 학업성적 등급이 C-(1.7) 이상이면 이수학점에 가산된다. 제21조의2(학사경고) ① 매 학기
---
[3] ② <삭제>③ 학위과정별, 대상자별 수료학점 인정에 대한 세부 사항은 따로 정한다.제18조 <삭제>제19조(수강과목) ① 학생은 소정의 교과과정에 의하여 개설된 수강과목을 선택할 수 있다.② 학생은 대면과목 이외에도 원격과목을 수강할 수 있으며, 이에 대한 세부 사항은 따로 정한다.③ 
---


---
## 3. Multi-Query
* 질문을 **2~3개 버전**으로 만들어 각각 검색한 뒤 합칩니다.
* 버전이 너무 많으면 의도가 표류(drift)하므로 오늘은 **3개**만 씁니다.
* 같은 chunk가 여러 번 나오면 한 번만 남깁니다.



In [16]:
def multi_queries(question: str, n: int = 3) -> list[str]:
    prompt = (
        f'학칙 검색용으로 아래 질문을 서로 다른 표현 {n}개로 다시 쓰세요.\n'
        '번호 없이, 한 줄에 하나씩만 출력하세요.\n'
        '설비명·학위과정 등 핵심 단어는 유지하세요.\n\n'
        f'질문: {question}'
    )
    lines = [ln.strip(' -•\t') for ln in llm.invoke(prompt).content.splitlines()]
    return [ln for ln in lines if ln][:n]


queries = multi_queries(RAW_Q, n=3)
for i, q in enumerate(queries, 1):
    print(f'{i}. {q}')

1. 석사 과정 졸업을 위해서는 대체로 몇 년을 다녀야 하나요?
2. 석사 학위를 받기 위해 필요한 최소 재학 기간은 얼마인가요?
3. 석사 과정에서 졸업하기 위해서는 보통 몇 년이 필요합니까?


In [17]:
def retrieve_multi(queries: list[str], k: int = 2, level: str = '대학원') -> list:
    seen = set()
    merged = []
    for q in queries:
        hits = vectorstore.similarity_search(q, k=k, filter={'level': level})
        print(f'· "{q}" → {len(hits)}건')
        for d in hits:
            key = (d.metadata.get('source_file'), d.metadata.get('page'), d.page_content[:80])
            if key in seen:
                continue
            seen.add(key)
            merged.append(d)
    return merged


multi_docs = retrieve_multi(queries)
print('\n통합(중복 제거) 후:', len(multi_docs), '건')
for i, d in enumerate(multi_docs, 1):
    print(f'[{i}]', d.page_content[:120].replace('\n', ' '))
    print('---')



· "석사 과정 졸업을 위해서는 대체로 몇 년을 다녀야 하나요?" → 2건
· "석사 학위를 받기 위해 필요한 최소 재학 기간은 얼마인가요?" → 2건
· "석사 과정에서 졸업하기 위해서는 보통 몇 년이 필요합니까?" → 2건

통합(중복 제거) 후: 2 건
[1] 제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학
---
[2] 2월 이전에 입학한 학생은 본 학칙 제15조에 불구하고 석사학위과정의 경우 24학점 이상 취득하여야 한다.2. 1974년 2월 이전에 소요학점의 이수를 필하고 2학기 이상 연구등록을 한 학생은 본 학칙 제28조에 불
---


---
## 4. CRAG식 3단계 판단
| 판정 | 의미 | 다음에 할 일(개념) |
|------|------|-------------------|
| Correct | 질문에 바로 답이 됨 | 문서 그대로 사용 |
| Ambiguous | 관련은 있으나 부분적 | 일부만 쓰거나 보강 |
| Incorrect | 거의 무관 | 버리고 웹검색 등 |



In [18]:
from typing import Literal

Verdict = Literal['Correct', 'Ambiguous', 'Incorrect']


def crag_judge(question: str, doc_text: str) -> Verdict:
    prompt = (
        '질문과 문서의 관계를 하나만 고르세요.\n'
        '- Correct: 문서만으로 질문에 답할 수 있음\n'
        '- Ambiguous: 관련은 있으나 핵심 답이 부족하거나 다른 내용과 섞임\n'
        '- Incorrect: 거의 무관\n'
        '단어 하나만 출력하세요.\n\n'
        f'질문: {question}\n문서: {doc_text[:800]}'
    )
    raw = llm.invoke(prompt).content.strip()
    for v in ('Correct', 'Ambiguous', 'Incorrect'):
        if v.lower() in raw.lower():
            return v  # type: ignore
    return 'Ambiguous'


# Multi-Query 상위 문서에 대해 판정
for i, d in enumerate(multi_docs[:4], 1):
    v = crag_judge(rewritten, d.page_content)
    snippet = d.page_content[:90].replace('\n', ' ')
    print(f'[{i}] {v} | {snippet}')



[1] Correct | 제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35
[2] Ambiguous | 2월 이전에 입학한 학생은 본 학칙 제15조에 불구하고 석사학위과정의 경우 24학점 이상 취득하여야 한다.2. 1974년 2월 이전에 소요학점의 이수를 필하고 2


In [19]:
# Correct만 남기기 (Ambiguous는 참고용으로 따로)
correct, ambiguous, incorrect = [], [], []
for d in multi_docs:
    v = crag_judge(rewritten, d.page_content)
    if v == 'Correct':
        correct.append(d)
    elif v == 'Ambiguous':
        ambiguous.append(d)
    else:
        incorrect.append(d)

print('Correct   :', len(correct))
print('Ambiguous :', len(ambiguous))
print('Incorrect :', len(incorrect))



Correct   : 1
Ambiguous : 1
Incorrect : 0


### 답변 생성
* Correct 문서가 있으면 그것만 근거로 답합니다.
* 없으면 Ambiguous를 쓰고, 그것도 없으면 "문서만으로 답하기 어렵다"고 말합니다.



In [20]:
def generate_answer(question: str, docs: list) -> str:
    if not docs:
        return '관련 학칙 근거를 충분히 찾지 못했습니다.'
    context = '\n\n'.join(d.page_content for d in docs[:3])
    prompt = (
        '아래 학칙 발췌만 근거로 질문에 두세 문장으로 답하세요.\n'
        '근거에 없으면 추측하지 마세요.\n\n'
        f'문서:\n{context}\n\n질문: {question}'
    )
    return llm.invoke(prompt).content


docs_for_answer = correct or ambiguous
print(generate_answer(RAW_Q, docs_for_answer))



석사학위과정 학생은 입학일로부터 4년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 제출해야 하므로, 대충 4년 정도 다녀야 졸업할 수 있습니다.


---
## 5. Contextual Retrieval 맛보기
* Chunk 앞에 원본 맥락 한 줄을 붙여 두면 검색이 쉬워집니다.


In [21]:
sample = multi_docs[0]
context_line = llm.invoke(
    '다음 학칙 발췌가 어떤 규정(학부/대학원, 주제)인지 '
    '한국어로 한 문장(40자 내외)만 쓰세요.\n\n' + sample.page_content[:600]
).content.strip()

plain = sample.page_content
contextualized = f'[맥락] {context_line}\n{plain}'

print('맥락 문장:', context_line)
print('\n--- 임베딩에 넣을 텍스트(앞부분) ---')
print(contextualized[:250])



맥락 문장: 대학원 학칙의 석사 및 박사학위 논문 심사 규정입니다.

--- 임베딩에 넣을 텍스트(앞부분) ---
[맥락] 대학원 학칙의 석사 및 박사학위 논문 심사 규정입니다.
제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에
